In [1]:
#install libaries
import numpy as np
import pandas as pd 
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
from sklearn.metrics.pairwise import cosine_similarity
import time
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer


/Users/daisy/anaconda3/envs/search-engine-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
#api keys
google_api_key = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=google_api_key)

In [2]:
#fixture data
user_queries = [
    "rare batman comics from the 80s",
    "first edition pokemon cards",
    "vintage vinyl jazz albums",
    "antique silver coins pre-1900",
    "limited edition hot wheels cars",
    "signed baseball memorabilia",
    "star wars action figures 1977",
    "graded magic the gathering cards",
    "mint condition barbie dolls",
    "beatles abbey road vinyl"
]

product_listings = [
    {
        "title": "Batman #404 Comic - Year One Part 1 (1987)",
        "description": "Classic Batman comic by Frank Miller, first issue in the 'Year One' storyline. Near mint condition, perfect for DC collectors."
    },
    {
        "title": "Charizard Holo 1st Edition - PSA 9",
        "description": "Iconic Pokémon card from the Base Set. Highly collectible, graded PSA 9 for condition. A grail for any TCG fan."
    },
    {
        "title": "Miles Davis - Kind of Blue (1959 Vinyl)",
        "description": "Vintage Columbia 6-eye pressing of the legendary jazz album. Slight wear on cover, vinyl is VG+."
    },
    {
        "title": "1904 Morgan Silver Dollar - Uncirculated",
        "description": "Beautiful example of a Morgan Dollar, Philadelphia Mint. Uncirculated with high detail on Liberty's face."
    },
    {
        "title": "Hot Wheels 1995 Treasure Hunt - '57 Chevy",
        "description": "Rare 1995 Treasure Hunt series car with original packaging. Highly sought-after piece from the early TH lineup."
    },
    {
        "title": "Derek Jeter Signed Yankees Jersey",
        "description": "Official MLB jersey signed by Derek Jeter with COA. Great addition to any sports memorabilia collection."
    },
    {
        "title": "Kenner Star Wars Luke Skywalker (1978)",
        "description": "Original action figure with telescoping lightsaber. Shows minor wear, but great articulation and paint detail."
    },
    {
        "title": "MTG Black Lotus - Proxy for Display",
        "description": "High-quality display proxy of the Black Lotus card from Alpha set. For aesthetic purposes only, not tournament legal."
    },
    {
        "title": "Barbie - 1985 Day-to-Night Edition",
        "description": "Vintage Barbie in both business and evening outfits. Excellent condition with original box."
    },
    {
        "title": "The Beatles - Abbey Road (1969 UK Pressing)",
        "description": "Original UK pressing with Apple label. Minor sleeve wear, but vinyl plays beautifully. Essential for any Beatles collection."
    }
]


**Gemini Embedding Experimental**

Experiemental, ranked first in Hugging Face MTEB leaderboard. Last updated in March 2025

Product listing (title + description), task type: RETRIEVAL_DOCUMENT

User query (search), task type: RETRIEVAL_QUERY

Requests Per Minute: 5
Requests Per Day: 100

In [ ]:
#test
result1 = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents="How does alphafold work?",
)

print(result1.embeddings)

#similarilty serach - Retrieval Augmented Generation (RAG)
result2 = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents="What is the meaning of life?",
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)
print(result2.embeddings)

[ContentEmbedding(values=[0.025964916, 0.0035924788, 0.021235455, -0.07027492, -0.04032638, 0.0226563, 0.014579744, -0.016282374, 0.023600742, 0.025808478, 0.009968264, 0.007745559, 0.021928139, 0.034615964, 0.12810431, 0.0019777848, -0.029766846, 0.039229263, 0.00694003, -0.019936852, -0.011004835, 0.027577255, -0.022434842, -0.009105839, 0.016378663, -0.011937824, 0.0077806795, 0.0041419356, 0.05548173, 0.0139697315, 0.0008025475, 0.0020434142, -0.014414456, -0.0010054252, -0.013071314, 0.024704844, -0.027062543, 0.0017097362, -0.0116361445, -0.012696176, -0.014259888, 0.011409137, 0.018161539, -0.02078812, 0.012737334, -0.041001767, -0.002706777, -0.034322426, -0.0102661885, -0.0013434804, -0.023296451, 0.0031671633, -0.00077531545, -0.17570695, 0.01176866, 0.02325038, 0.013082423, -0.028667854, -0.0015029972, 0.014727239, -0.010287542, -0.00035226718, -0.008449122, 0.0043781595, 0.017853355, -0.022493502, -0.00028810848, -0.023935927, -0.025332574, 0.0014073389, -0.0028375483, 0.00

In [14]:
def embed_text(text, task_type):
    response = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return response.embeddings

In [69]:
product_embeddings = []
for item in product_listings:
    text = f"{item['title']} {item['description']}"
    vec = embed_text(text, task_type="RETRIEVAL_DOCUMENT")
    if vec:
        product_embeddings.append(vec)
    time.sleep(15)

query_embeddings = []
for query in user_queries:
    vec = embed_text(query, task_type="RETRIEVAL_QUERY")
    if vec:
        query_embeddings.append(vec)
    time.sleep(15)

In [70]:
print(product_embeddings)
print(product_embeddings[0])
print(product_embeddings[0][0].values)
print(np.array(product_embeddings).shape)
print(np.array(query_embeddings).shape)

[[ContentEmbedding(values=[-0.026368296, -0.008535967, 0.026480205, -0.055373915, -0.00597067, 0.03155749, -0.017350594, -0.00010059426, 0.0007399367, -0.0021323191, -0.007055468, -0.00861895, -0.013163686, 0.02430289, 0.09904903, -0.02564834, -0.0037714697, -0.012548553, 0.008095185, -0.03262026, -0.0032981576, -0.022020767, -0.00046809803, 0.0072678444, 0.023347441, 0.01741853, 0.005973042, 0.035775695, -0.0058169602, 0.008563488, 0.004334334, 0.032336872, 0.018311879, -0.018816765, 0.011121296, 0.01892934, -0.008683328, -0.002209379, -0.0015621348, -0.0102565, 0.005420591, 0.018386645, 0.02056993, 0.0076362244, -0.0096865315, 0.015053191, 0.015766477, 0.003696759, -0.016252726, 0.0026976338, -0.013897975, -0.0046554534, -0.01842877, -0.1983632, -0.0051338933, 0.021392282, 0.019705908, -0.0058917827, -0.016766494, -0.009119785, -0.003085341, 0.017956905, -0.01734602, -0.0416818, -0.03266811, 0.00793509, 0.025751796, 0.00274249, -0.0041472865, -0.037636995, 0.01946998, 0.0023814072, -

In [73]:
product_embeddings_np = np.array([item[0].values for item in product_embeddings])  # Extract the 'values' from each embedding object
query_embeddings_np = np.array([query_vec[0].values for query_vec in query_embeddings])  # Extract the 'values' from each query embedding

for i, query_vec in enumerate(query_embeddings_np):
    query_vec = query_vec.reshape(1, -1)  # Shape: (1, D)
    scores = cosine_similarity(query_vec, product_embeddings_np)[0]  # Shape: (N,)
    top_indices = scores.argsort()[::-1][:3]

    print(f"\nQuery: {user_queries[i]}")
    for rank, idx in enumerate(top_indices, start=1):
        product = product_listings[idx]
        print(f"{rank}. {product['title']} (score: {scores[idx]:.4f})")


Query: rare batman comics from the 80s
1. Batman #404 Comic - Year One Part 1 (1987) (score: 0.7286)
2. Charizard Holo 1st Edition - PSA 9 (score: 0.6064)
3. Barbie - 1985 Day-to-Night Edition (score: 0.5977)

Query: first edition pokemon cards
1. Charizard Holo 1st Edition - PSA 9 (score: 0.7247)
2. MTG Black Lotus - Proxy for Display (score: 0.6348)
3. Hot Wheels 1995 Treasure Hunt - '57 Chevy (score: 0.6055)

Query: vintage vinyl jazz albums
1. Miles Davis - Kind of Blue (1959 Vinyl) (score: 0.7234)
2. The Beatles - Abbey Road (1969 UK Pressing) (score: 0.6379)
3. Barbie - 1985 Day-to-Night Edition (score: 0.5831)

Query: antique silver coins pre-1900
1. 1904 Morgan Silver Dollar - Uncirculated (score: 0.6798)
2. Hot Wheels 1995 Treasure Hunt - '57 Chevy (score: 0.5821)
3. Miles Davis - Kind of Blue (1959 Vinyl) (score: 0.5788)

Query: limited edition hot wheels cars
1. Hot Wheels 1995 Treasure Hunt - '57 Chevy (score: 0.6945)
2. Barbie - 1985 Day-to-Night Edition (score: 0.6075)
3

**Linq-AI-Research/Linq-Embed-Mistral**

Ranked 2nd in the MTEB. Developed on  E5-mistral-7b-instruct and Mistral-7B-v0.1 models. Focus on imporoving text retrieval.

Open source

In [3]:
# sentence-transformers library
st_model = SentenceTransformer("Linq-AI-Research/Linq-Embed-Mistral")

Loading checkpoint shards: 100%|██████████| 3/3 [01:01<00:00, 20.36s/it]


KeyboardInterrupt: 

In [ ]:
#continuing with sentence-transformers library, encode user queries & product listisngs
product_texts = [f"{item['title']} {item['description']}" for item in product_listings]
product_embeddings_lem_st = st_model.encode(product_texts, convert_to_numpy=True)

query_embeddings_lem_st = st_model.encode(user_queries, convert_to_numpy=True)

In [ ]:
# compare the vectors

In [ ]:
# transformer libary
tokenizer = AutoTokenizer.from_pretrained("Linq-AI-Research/Linq-Embed-Mistral")
model = AutoModel.from_pretrained("Linq-AI-Research/Linq-Embed-Mistral")

Loading checkpoint shards: 100%|██████████| 3/3 [01:06<00:00, 22.08s/it]


In [ ]:
# transformer library
def embed_text_LEM(text):
    inputs = tokenizer(        # converts raw text into model ready format
        text,
        return_tensors="pt",   # return PyTorch tensors
        truncation=True,       # Make sure long text gets clipped
        max_length=512         # You can increase this if needed (1024?)
    )
    
    # disables gradient calculations for faster inference
    with torch.no_grad():
        outputs = model(**inputs) # feeds the tokenized inputs into the model, outputs last_hidden_state
                                  # which is a tensor of shape  
    
    print(outputs)
    
    # This assumes the embedding is in the last_hidden_state and you average it
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [ ]:
# transformer library
query_embeddings_LEM = [embed_text_LEM(query) for query in user_queries]
product_embeddings_LEM = [embed_text_LEM(f"{item['title']} {item['description']}") for item in product_listings]

KeyboardInterrupt: 

In [ ]:
for i, query_vec in enumerate(query_embeddings_LEM):
    query_vec = query_vec.reshape(1, -1)
    product_matrix = np.array(product_embeddings)
    scores = cosine_similarity(query_vec, product_matrix)[0]
    top_indices = scores.argsort()[::-1][:3]

    print(f"\nQuery: {user_queries[i]}")
    for rank, idx in enumerate(top_indices, start=1):
        product = product_listings[idx]
        print(f"{rank}. {product['title']} (score: {scores[idx]:.4f})")

*insert here a place to test a vector database*